# 🏗 PHASE 2 — STEP 6: ML Engineering Structure
## Enterprise ML Discipline — Crop Intelligence Platform

This notebook establishes a **controlled, reproducible, config-driven ML pipeline** with:
- Strict separation of **research vs production**
- **Artifact discipline** (zero Git bloat)
- **Config-driven training** (YAML parameters)
- Future **MLflow / Docker** compatibility

> ⚠️ This notebook is a **research tool only**. Production logic lives in `training/`.

## 1. Setup ML Directory Structure

Programmatically create and verify the full ML directory tree.

In [ ]:
"""Section 1: Create and verify the ML directory structure."""
import os
from pathlib import Path

# ML root is one level up from notebooks/
ML_ROOT = Path(os.path.abspath("")).parent  # resolves to ml/

REQUIRED_DIRS = [
    "notebooks",   # Research only — EDA, feature exploration, visualization
    "data",        # Raw + processed datasets (Git-ignored)
    "models",      # Trained model artifacts (Git-ignored)
    "training",    # Production training scripts
    "utils",       # Shared ML utilities
    "configs",     # YAML/JSON experiment configs
    "mlflow",      # Future MLflow integration
]

# Create directories (idempotent)
for d in REQUIRED_DIRS:
    (ML_ROOT / d).mkdir(parents=True, exist_ok=True)

# Print tree representation
print(f"ML Root: {ML_ROOT}\n")
print("ml/")
for d in sorted(ML_ROOT.iterdir()):
    if d.is_dir() and not d.name.startswith((".", "venv", "__")):
        print(f"├── {d.name}/")
        for f in sorted(d.iterdir()):
            if not f.name.startswith("."):
                print(f"│   ├── {f.name}")
    elif d.is_file() and not d.name.startswith("."):
        print(f"├── {d.name}")

# Validate all required dirs exist
missing = [d for d in REQUIRED_DIRS if not (ML_ROOT / d).is_dir()]
assert not missing, f"Missing directories: {missing}"
print("\n✅ All 7 required directories present.")

## 2. Create ML-Specific .gitignore

Ensure datasets, model artifacts, and Jupyter checkpoints never enter Git.

In [ ]:
"""Section 2: Create and verify ml/.gitignore."""

GITIGNORE_CONTENT = """\
# ── Datasets ──────────────────────────────────────
data/
*.csv
*.parquet
*.h5
*.npz

# ── Model artifacts ───────────────────────────────
models/
*.pt
*.pth
*.pkl
*.joblib

# ── Jupyter ───────────────────────────────────────
.ipynb_checkpoints

# ── Environment ───────────────────────────────────
venv/

# ── OS ────────────────────────────────────────────
.DS_Store
"""

gitignore_path = ML_ROOT / ".gitignore"
gitignore_path.write_text(GITIGNORE_CONTENT, encoding="utf-8")

# Read back and display
print("── ml/.gitignore ──")
print(gitignore_path.read_text(encoding="utf-8"))

# Verify key patterns are present
content = gitignore_path.read_text(encoding="utf-8")
for pattern in ["data/", "models/", "*.pkl", "*.pt", ".ipynb_checkpoints", "venv/"]:
    assert pattern in content, f"Missing pattern: {pattern}"

print("✅ .gitignore validated — data/, models/, checkpoints, and venv/ are excluded.")

## 3. Define ML Boundaries in README

Document what the ML module **does** and **does not** contain.

In [ ]:
"""Section 3: Generate and display ml/README.md with strict boundaries."""

README_CONTENT = """\
# ML Module — Crop Intelligence Platform

## Purpose
- Model training & evaluation
- Experimentation & feature engineering
- Model artifact generation
- Config-driven hyperparameter management

## Does NOT Contain
- API logic (lives in `backend/`)
- Frontend code (lives in `frontend/`)
- Docker configuration (lives in `docker/`)

## Directory Layout
| Directory    | Purpose                                       |
|-------------|-----------------------------------------------|
| `notebooks/` | Research only — EDA, visualization, exploration |
| `data/`      | Raw + processed datasets (Git-ignored)         |
| `models/`    | Trained model artifacts (Git-ignored)          |
| `training/`  | Production training scripts                    |
| `utils/`     | Shared ML utilities                            |
| `configs/`   | YAML/JSON experiment configs                   |
| `mlflow/`    | Future MLflow integration                      |

## Rules
1. Research notebooks are **exploratory only**
2. Production logic **must** live in `training/`
3. Notebooks are **disposable** research tools
4. Never save model weights from notebooks
5. Never hardcode file paths — use configs
6. All training must be **config-driven** (YAML)

## Quick Start
```bash
cd ml
python -m venv venv
source venv/bin/activate  # or .\\venv\\Scripts\\Activate.ps1
pip install -r requirements.txt
python training/train_xgboost.py
```
"""

readme_path = ML_ROOT / "README.md"
readme_path.write_text(README_CONTENT, encoding="utf-8")

print("── ml/README.md ──")
print(readme_path.read_text(encoding="utf-8"))
print("✅ README with enterprise boundaries created.")

## 4. Create Config-Driven Training Configuration

Enterprise ML is always **config-driven** — no magic numbers in code.

In [ ]:
"""Section 4: Create and display configs/xgboost.yaml."""
import yaml

CONFIG = {
    "model": {
        "n_estimators": 100,
        "max_depth": 6,
        "learning_rate": 0.1,
        "objective": "reg:squarederror",
        "random_state": 42,
    },
    "data": {
        "input_path": "data/sample.csv",
        "target_column": "yield",
        "test_size": 0.2,
    },
    "output": {
        "model_path": "models/xgb_model.pkl",
        "metrics_path": "models/metrics.json",
    },
}

config_path = ML_ROOT / "configs" / "xgboost.yaml"
with open(config_path, "w", encoding="utf-8") as f:
    yaml.dump(CONFIG, f, default_flow_style=False, sort_keys=False)

# Load back and display
with open(config_path, encoding="utf-8") as f:
    loaded = yaml.safe_load(f)

print("── configs/xgboost.yaml ──")
print(yaml.dump(loaded, default_flow_style=False, sort_keys=False))

print("Why config-driven?")
print("  • Reproducibility — exact params are version-controlled")
print("  • Experiment comparison — swap YAML files, not code")
print("  • Parameter control — no magic numbers buried in scripts")
print("\n✅ Config file created and validated.")

## 5. Build the XGBoost Training Entry Point

Production training scripts live in `training/`, **not** notebooks.

In [ ]:
"""Section 5: Create training/train_xgboost.py and display it."""

TRAIN_SCRIPT = '''\
"""
XGBoost Training Script — Crop Intelligence Platform
Config-driven, reproducible, production-grade.
"""
import json
import sys
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import yaml
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from xgboost import XGBRegressor


def load_config(config_path: str = "configs/xgboost.yaml") -> dict:
    """Load experiment configuration from YAML."""
    with open(config_path, encoding="utf-8") as f:
        return yaml.safe_load(f)


def load_data(config: dict) -> pd.DataFrame:
    """Load dataset from path specified in config. Falls back to synthetic data."""
    data_path = Path(config["data"]["input_path"])
    if data_path.exists():
        print(f"Loading data from {data_path}")
        return pd.read_csv(data_path)

    print("⚠ No dataset found — generating synthetic data for validation.")
    rng = np.random.default_rng(42)
    n = 500
    return pd.DataFrame({
        "temperature": rng.uniform(15, 40, n),
        "rainfall": rng.uniform(200, 1200, n),
        "soil_ph": rng.uniform(5.5, 8.0, n),
        "nitrogen": rng.uniform(20, 120, n),
        "yield": rng.uniform(1.5, 6.5, n),
    })


def train(config: dict) -> None:
    """Train XGBoost model with config-driven parameters."""
    df = load_data(config)
    target = config["data"]["target_column"]

    X = df.drop(columns=[target])
    y = df[target]

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=config["data"]["test_size"], random_state=42
    )

    model = XGBRegressor(**config["model"])
    model.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)

    y_pred = model.predict(X_test)
    metrics = {
        "r2": round(float(r2_score(y_test, y_pred)), 4),
        "mae": round(float(mean_absolute_error(y_test, y_pred)), 4),
        "rmse": round(float(np.sqrt(mean_squared_error(y_test, y_pred))), 4),
    }

    print(f"Metrics: R²={metrics['r2']}, MAE={metrics['mae']}, RMSE={metrics['rmse']}")

    # Save artifacts
    model_path = Path(config["output"]["model_path"])
    model_path.parent.mkdir(parents=True, exist_ok=True)
    joblib.dump(model, model_path)
    print(f"Model saved → {model_path}")

    metrics_path = Path(config["output"]["metrics_path"])
    with open(metrics_path, "w", encoding="utf-8") as f:
        json.dump(metrics, f, indent=2)
    print(f"Metrics saved → {metrics_path}")


if __name__ == "__main__":
    cfg_path = sys.argv[1] if len(sys.argv) > 1 else "configs/xgboost.yaml"
    cfg = load_config(cfg_path)
    train(cfg)
'''

train_path = ML_ROOT / "training" / "train_xgboost.py"
train_path.write_text(TRAIN_SCRIPT, encoding="utf-8")

# Create __init__.py
init_path = ML_ROOT / "training" / "__init__.py"
if not init_path.exists():
    init_path.write_text("", encoding="utf-8")

print("── training/train_xgboost.py ──")
print(TRAIN_SCRIPT)
print("\nFunction responsibilities:")
print("  • load_config()  — reads YAML experiment parameters")
print("  • load_data()    — loads CSV or generates synthetic fallback")
print("  • train()        — split → fit → evaluate → save artifacts")
print("\n✅ Training entry point created.")

## 6. Build Shared ML Utilities

Reusable helpers prevent code duplication across training scripts and notebooks.

In [ ]:
"""Section 6: Create utils/helpers.py with reusable ML functions."""

HELPERS_CODE = '''\
"""Shared ML utilities — Crop Intelligence Platform."""
import json
import logging
from pathlib import Path

import joblib
import numpy as np
import yaml
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


def load_config(config_path: str) -> dict:
    """Load YAML experiment configuration."""
    with open(config_path, encoding="utf-8") as f:
        return yaml.safe_load(f)


def setup_logging(level: str = "INFO") -> logging.Logger:
    """Configure standardized ML logging."""
    logger = logging.getLogger("crop-ml")
    logger.setLevel(getattr(logging, level.upper(), logging.INFO))
    if not logger.handlers:
        handler = logging.StreamHandler()
        fmt = logging.Formatter("[%(asctime)s] %(levelname)s — %(message)s", datefmt="%H:%M:%S")
        handler.setFormatter(fmt)
        logger.addHandler(handler)
    return logger


def save_model(model, path: str) -> None:
    """Save model artifact using joblib."""
    p = Path(path)
    p.parent.mkdir(parents=True, exist_ok=True)
    joblib.dump(model, p)


def load_model(path: str):
    """Load model artifact from disk."""
    return joblib.load(path)


def evaluate_model(y_true, y_pred) -> dict:
    """Compute standard regression metrics: RMSE, MAE, R²."""
    return {
        "r2": round(float(r2_score(y_true, y_pred)), 4),
        "mae": round(float(mean_absolute_error(y_true, y_pred)), 4),
        "rmse": round(float(np.sqrt(mean_squared_error(y_true, y_pred))), 4),
    }
'''

# Write helpers.py
helpers_path = ML_ROOT / "utils" / "helpers.py"
helpers_path.write_text(HELPERS_CODE, encoding="utf-8")

# Write __init__.py
init_path = ML_ROOT / "utils" / "__init__.py"
init_path.write_text('from .helpers import load_config, setup_logging, save_model, load_model, evaluate_model\n', encoding="utf-8")

print("── utils/helpers.py ──")
print(HELPERS_CODE)

# Demonstrate imports work
print("Available utilities:")
print("  • load_config()    — Read YAML configs")
print("  • setup_logging()  — Standardized ML logging")
print("  • save_model()     — Artifact persistence wrapper")
print("  • load_model()     — Artifact loading wrapper")
print("  • evaluate_model() — RMSE, MAE, R² computation")
print("\n✅ Shared utilities created.")

## 7. Enforce Notebook Discipline with a Template

Notebooks are **disposable research tools**. Rules:
- Only EDA, feature exploration, visualization, temporary experiments
- **Never** save model weights from notebooks
- **Never** hardcode file paths — use configs
- **Never** call API directly

In [ ]:
"""Section 7: Create EDA notebook template using nbformat."""
import json

EDA_NOTEBOOK = {
    "cells": [
        {
            "cell_type": "markdown",
            "metadata": {},
            "source": [
                "# EDA Template — Crop Intelligence Platform\\n",
                "\\n",
                "## Rules\\n",
                "- **Only** EDA, feature exploration, visualization, temporary experiments\\n",
                "- **Never** save model weights from notebooks\\n",
                "- **Never** hardcode file paths — use configs\\n",
                "- **Never** call API directly\\n",
                "- Notebooks are **disposable** research tools\\n",
            ],
        },
        {
            "cell_type": "code",
            "metadata": {},
            "source": [
                "import pandas as pd\\n",
                "import numpy as np\\n",
                "import matplotlib.pyplot as plt\\n",
                "import seaborn as sns\\n",
                "\\n",
                "# Config-driven data path\\n",
                "import yaml\\n",
                "with open('../configs/xgboost.yaml') as f:\\n",
                "    config = yaml.safe_load(f)\\n",
                "\\n",
                "DATA_PATH = config['data']['input_path']\\n",
                "print(f'Data path: {DATA_PATH}')\\n",
            ],
            "execution_count": None,
            "outputs": [],
        },
        {
            "cell_type": "markdown",
            "metadata": {},
            "source": ["## Load & Profile Data\\n"],
        },
        {
            "cell_type": "code",
            "metadata": {},
            "source": [
                "# Load data (replace with real dataset)\\n",
                "# df = pd.read_csv(f'../{DATA_PATH}')\\n",
                "\\n",
                "# Synthetic fallback for EDA template\\n",
                "rng = np.random.default_rng(42)\\n",
                "n = 500\\n",
                "df = pd.DataFrame({\\n",
                "    'temperature': rng.uniform(15, 40, n),\\n",
                "    'rainfall': rng.uniform(200, 1200, n),\\n",
                "    'soil_ph': rng.uniform(5.5, 8.0, n),\\n",
                "    'nitrogen': rng.uniform(20, 120, n),\\n",
                "    'yield': rng.uniform(1.5, 6.5, n),\\n",
                "})\\n",
                "\\n",
                "print(df.info())\\n",
                "df.describe()\\n",
            ],
            "execution_count": None,
            "outputs": [],
        },
        {
            "cell_type": "markdown",
            "metadata": {},
            "source": ["## Distribution Plots\\n"],
        },
        {
            "cell_type": "code",
            "metadata": {},
            "source": [
                "fig, axes = plt.subplots(1, 2, figsize=(12, 4))\\n",
                "\\n",
                "sns.histplot(df['yield'], kde=True, ax=axes[0], color='#10B981')\\n",
                "axes[0].set_title('Yield Distribution')\\n",
                "\\n",
                "sns.boxplot(data=df[['temperature', 'rainfall', 'soil_ph', 'nitrogen']], ax=axes[1])\\n",
                "axes[1].set_title('Feature Distributions')\\n",
                "\\n",
                "plt.tight_layout()\\n",
                "plt.show()\\n",
            ],
            "execution_count": None,
            "outputs": [],
        },
        {
            "cell_type": "markdown",
            "metadata": {},
            "source": ["## Correlation Heatmap\\n"],
        },
        {
            "cell_type": "code",
            "metadata": {},
            "source": [
                "plt.figure(figsize=(8, 6))\\n",
                "sns.heatmap(df.corr(), annot=True, cmap='RdYlGn', center=0, fmt='.2f')\\n",
                "plt.title('Feature Correlation Matrix')\\n",
                "plt.tight_layout()\\n",
                "plt.show()\\n",
            ],
            "execution_count": None,
            "outputs": [],
        },
    ],
    "metadata": {
        "kernelspec": {"display_name": "Python 3", "language": "python", "name": "python3"},
        "language_info": {"name": "python", "version": "3.12.0"},
    },
    "nbformat": 4,
    "nbformat_minor": 5,
}

template_path = ML_ROOT / "notebooks" / "01_eda_template.ipynb"
with open(template_path, "w", encoding="utf-8") as f:
    json.dump(EDA_NOTEBOOK, f, indent=1)

print(f"✅ EDA template created at: {template_path}")
print(f"   Cells: {len(EDA_NOTEBOOK['cells'])} (4 code + 4 markdown)")
print("   Includes: imports, data profiling, distributions, correlation heatmap")

## 8. Setup Isolated Python Environment for ML

ML environment is **isolated** from backend — no cross-dependency pollution.

In [ ]:
"""Section 8: Generate ml/requirements.txt for isolated ML environment."""

REQUIREMENTS = """\
# ── Core ML Dependencies ─────────────────────────
pandas>=2.0.0
scikit-learn>=1.3.0
xgboost>=2.0.0
joblib>=1.3.0
pyyaml>=6.0

# ── Visualization ────────────────────────────────
matplotlib>=3.7.0
seaborn>=0.13.0

# ── Jupyter ──────────────────────────────────────
jupyter>=1.0.0
nbformat>=5.9.0

# ── Numeric ──────────────────────────────────────
numpy>=1.24.0
"""

req_path = ML_ROOT / "requirements.txt"
req_path.write_text(REQUIREMENTS, encoding="utf-8")

print("── ml/requirements.txt ──")
print(REQUIREMENTS)

# Verify venv/ is in .gitignore
gitignore = (ML_ROOT / ".gitignore").read_text(encoding="utf-8")
assert "venv/" in gitignore, "venv/ not in .gitignore!"

print("Setup commands:")
print("  cd ml")
print("  python -m venv venv")
print("  .\\venv\\Scripts\\Activate.ps1  # Windows")
print("  pip install -r requirements.txt")
print("\n✅ requirements.txt created. venv/ is Git-ignored.")

## 9. Add MLflow Placeholder Structure

Architecture-ready for future **experiment tracking**, **model registry**, and **metrics logging**.

In [ ]:
"""Section 9: Create MLflow placeholder directory and config."""

# MLflow README
mlflow_readme = ML_ROOT / "mlflow" / "README.md"
mlflow_readme.parent.mkdir(parents=True, exist_ok=True)
mlflow_readme.write_text("""\
# MLflow Integration — Crop Intelligence Platform

## Future Integration Plans
- **Experiment Tracking**: Log parameters, metrics, and artifacts per training run
- **Model Registry**: Version and stage models (staging → production)
- **Metrics Logging**: Track R², RMSE, MAE across experiments
- **Artifact Versioning**: Store and retrieve model files with lineage

## Status
🔲 Not yet active — placeholder for architecture readiness.
""", encoding="utf-8")

# MLflow config stub
mlflow_config = ML_ROOT / "mlflow" / "mlflow_config.yaml"
mlflow_config_content = {
    "tracking_uri": "http://localhost:5000",
    "experiment_name": "crop-yield-prediction",
    "artifact_location": "./mlflow/artifacts",
    "registered_model_name": "xgb-crop-yield",
}
with open(mlflow_config, "w", encoding="utf-8") as f:
    yaml.dump(mlflow_config_content, f, default_flow_style=False)

print("── mlflow/README.md ──")
print(mlflow_readme.read_text(encoding="utf-8"))
print("── mlflow/mlflow_config.yaml ──")
print(yaml.dump(mlflow_config_content, default_flow_style=False))
print("✅ MLflow placeholder created — architecture-ready.")

## 10. Validate Git Cleanliness and Commit

Verify no datasets or model artifacts are staged — only structure + scripts.

In [ ]:
"""Section 10: Validate git cleanliness — no datasets or model artifacts staged."""
import subprocess

DANGEROUS_EXTENSIONS = {".csv", ".parquet", ".h5", ".npz", ".pt", ".pth", ".pkl", ".joblib"}

# Simulate git add and status
print("── Git Validation ──")
print("Files that WOULD be committed (ml/ structure + scripts only):")
print()

committed_files = []
blocked_files = []

for p in ML_ROOT.rglob("*"):
    if p.is_file():
        rel = p.relative_to(ML_ROOT)
        parts = str(rel).replace("\\", "/")

        # Skip ignored directories
        if any(skip in parts for skip in ["venv/", "data/", "models/", ".ipynb_checkpoints"]):
            continue

        if p.suffix in DANGEROUS_EXTENSIONS:
            blocked_files.append(str(rel))
        else:
            committed_files.append(str(rel))

for f in sorted(committed_files):
    print(f"  ✅ {f}")

if blocked_files:
    print(f"\n  ❌ BLOCKED: {blocked_files}")
else:
    print(f"\n  No dangerous files detected.")

print(f"\n  Total files to commit: {len(committed_files)}")
print(f"  Blocked files: {len(blocked_files)}")
print(f'\n  Commit command:')
print(f'  git commit -m "chore(ml): establish structured ML training pipeline and git discipline"')

## 11. Run Full Success Checklist Validation

Programmatic pass/fail verification of every requirement.

In [ ]:
"""Section 11: Final success checklist — all checks must pass."""

checks = {}

# 1. ML folder structured correctly (7 subdirectories)
required = ["notebooks", "data", "models", "training", "utils", "configs", "mlflow"]
checks["ML folder structured correctly"] = all((ML_ROOT / d).is_dir() for d in required)

# 2. data/ ignored
gitignore = (ML_ROOT / ".gitignore").read_text(encoding="utf-8")
checks["data/ ignored"] = "data/" in gitignore

# 3. models/ ignored
checks["models/ ignored"] = "models/" in gitignore

# 4. Training script exists
checks["Training script exists"] = (ML_ROOT / "training" / "train_xgboost.py").is_file()

# 5. Config-based parameters ready
checks["Config-based parameters ready"] = (ML_ROOT / "configs" / "xgboost.yaml").is_file()

# 6. ML environment isolated
checks["ML environment isolated"] = (
    (ML_ROOT / "requirements.txt").is_file() and "venv/" in gitignore
)

# 7. No large files tracked (>1MB in non-ignored dirs)
large_files = []
for p in ML_ROOT.rglob("*"):
    if p.is_file() and p.stat().st_size > 1_000_000:
        rel = str(p.relative_to(ML_ROOT)).replace("\\", "/")
        if not any(skip in rel for skip in ["venv/", "data/", "models/"]):
            large_files.append(rel)
checks["No large files tracked"] = len(large_files) == 0

# 8. README created
checks["README created"] = (ML_ROOT / "README.md").is_file()

# 9. Shared utilities exist
checks["Shared utilities exist"] = (ML_ROOT / "utils" / "helpers.py").is_file()

# Print report
print("=" * 55)
print("  SUCCESS CHECKLIST — ML Engineering Structure")
print("=" * 55)

all_pass = True
for name, passed in checks.items():
    icon = "✅" if passed else "❌"
    status = "PASS" if passed else "FAIL"
    if not passed:
        all_pass = False
    print(f"  {icon} {status} — {name}")

print("=" * 55)

if all_pass:
    print("\n  🎉 ALL CHECKS PASSED")
    print("\n  What you built:")
    print("  ─────────────────────────────────────────────")
    print("  A Reproducible ML Engineering Layer supporting:")
    print("    • Clean experiments (notebooks/)")
    print("    • Controlled training (training/)")
    print("    • Config-driven params (configs/)")
    print("    • Artifact discipline (data/ + models/ ignored)")
    print("    • Shared utilities (utils/)")
    print("    • Future MLflow / CI integration (mlflow/)")
    print("    • Docker ML containers (requirements.txt)")
    print("\n  Architecture Maturity:")
    print("    Before: Placeholder ML folder")
    print("    Now:    Structured ML pipeline base")
else:
    print("\n  ⚠ Some checks failed — review above.")